# CIDA-Plugin × Frozen LLM on GSM8K

This notebook trains the **CIDA multi-agent deliberation plugin** on top of a fully-frozen small LLM (Qwen1.5-1.8B-Chat) for grade-school math reasoning (GSM8K).

**Key ideas**
- The LLM is a frozen feature extractor. Only CIDA weights are trained.
- CIDA agents deliberate over the hidden state of the last prompt token.
- The deliberated vector is projected back to the LLM vocabulary and cross-entropy loss is computed against the target answer tokens.
- We then compare Base-LLM vs Base-LLM+CIDA exact-match accuracy on GSM8K test.

**Runtime**: GPU (T4 or better). Set *Runtime → Change runtime type → T4 GPU* before running.

## 1 · Environment Setup

In [ ]:
# ─── Install dependencies ────────────────────────────────────────────────────
!pip install -q torch transformers datasets accelerate torchdiffeq
print("✅ Dependencies installed")

In [ ]:
import subprocess, sys, os

# ─── Clone CIDA-Plugin repository ───────────────────────────────────────────
repo_url  = "https://github.com/Kairatzh/CIDA-plugin.git"
repo_dir  = "CIDA-plugin"

if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", "--depth", "1", repo_url], check=True)
    print("✅ Repository cloned")
else:
    print("ℹ️  Repository already present, skipping clone")

# ─── Add the cida_plugin package to Python path ──────────────────────────────
# The package folder inside the repo is named 'cida_plugin'
sys.path.insert(0, os.path.abspath(repo_dir))
print("🐍 sys.path updated:", sys.path[0])

In [ ]:
import re, math, json, time
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# CIDA imports (from cloned repo)
from cida_plugin import CIDAPlugin, CIDAPluginConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥  Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 2 · Load Frozen LLM

We use **Qwen/Qwen1.5-1.8B-Chat** (1.8B params, hidden size 2048). All weights are permanently frozen — gradients will only flow through CIDA.

In [ ]:
# ─── Configuration ───────────────────────────────────────────────────────────
MODEL_NAME   = "Qwen/Qwen1.5-1.8B-Chat"
# Fallback if Qwen is unavailable: "google/gemma-2b-it" (hidden_size=2048 too)

MAX_Q_LEN    = 256   # max tokens for question prompt
MAX_A_LEN    = 64   # max new tokens for answer generation
TRAIN_SUBSET = 1500  # samples used for training (speed vs. accuracy)
BATCH_SIZE   = 4
EPOCHS       = 2
LR           = 3e-4

# ─── Tokenizer ───────────────────────────────────────────────────────────────
print(f"Loading tokenizer from {MODEL_NAME}…")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="left",   # causal LM: pad on left
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ─── Model ───────────────────────────────────────────────────────────────────
print(f"Loading model {MODEL_NAME}…")
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,   # fp16 to fit in T4 VRAM
    device_map="auto",
    output_hidden_states=True,   # ← we need hidden states
)

# ─── Freeze ALL LLM weights ──────────────────────────────────────────────────
for param in llm.parameters():
    param.requires_grad = False
llm.eval()   # BN/Dropout in eval mode (frozen backbone)

trainable = sum(p.numel() for p in llm.parameters() if p.requires_grad)
total     = sum(p.numel() for p in llm.parameters())
print(f"✅ LLM loaded: {total/1e6:.0f}M params, {trainable} trainable (should be 0)")

# Retrieve hidden size for CIDA
LLM_HIDDEN = llm.config.hidden_size
VOCAB_SIZE  = llm.config.vocab_size
print(f"   hidden_size={LLM_HIDDEN}, vocab_size={VOCAB_SIZE}")

## 3 · Instantiate CIDA Plugin

CIDA wraps the LLM's final hidden state and runs 3 rounds of multi-agent deliberation before feeding the refined vector back to the LM head.

In [ ]:
# ─── CIDA config ─────────────────────────────────────────────────────────────
# We treat the task as binary (correct / incorrect) but the output we
# actually use is the continuous refined hidden state, not the CIDA belief.
# For the loss we use num_classes=2 (Prosecutor vs Defender framing).
cida_cfg = CIDAPluginConfig(
    d_input               = LLM_HIDDEN,   # match LLM hidden dim
    d_hidden              = 512,           # internal CIDA dim (smaller for speed)
    d_message             = 256,
    num_agents            = 4,
    num_classes           = 2,
    max_rounds            = 3,
    multi_label           = False,
    num_attn_heads        = 4,
    role_prior_blend      = 0.30,
    lambda_cal            = 0.3,
    lambda_ac             = 0.2,
    comm_dropout          = 0.1,
    use_perspective_projector = True,
    anonymous_messages    = True,
    use_dynamic_trust     = False,
    use_ttt               = False,         # disabled for speed
    use_liquid_dynamics   = False,         # discrete rounds for stability
    early_stop_threshold  = 0.92,
    sparse_comm_k         = 2,
)

cida = CIDAPlugin(cida_cfg).to(DEVICE).float()   # fp32 for stable training

# Verify only CIDA has grad
cida_params = sum(p.numel() for p in cida.parameters() if p.requires_grad)
print(f"✅ CIDA: {cida_params/1e6:.2f}M trainable parameters")

## 4 · Latent Plan Projector (CIDAPlanProjector)

This module converts CIDA's multi-agent refined states into **virtual plan tokens**
that are prepended to the answer as soft-prompt embeddings.

```
state_final.s : (B, M=4, d_hidden=512)
  └─ flatten  → (B, M * d_hidden = 2048)
  └─ Linear   → (B, num_plan_tokens * d_llm)   [num_plan_tokens=4]
  └─ reshape  → (B, 4, d_llm=2048)
```

The LLM embedding layer is used **read-only** to convert prompt / answer token IDs
into continuous embeddings; it is never updated.

In [ ]:
# ── CIDAPlanProjector ───────────────────────────────────────────────────────
# Simplified projector: single Linear + LayerNorm.
# Reduces trainable params from ~42M to ~16M, preventing
# overfitting on small datasets and saving VRAM on T4.

class CIDAPlanProjector(nn.Module):
    """
    Projects CIDA state_final.s → virtual soft-prompt plan tokens.
    Input  : (B, M, d_cida)
    Output : (B, num_plan_tokens, d_llm)
    """

    def __init__(self, d_cida, d_llm, num_agents, num_plan_tokens=4):
        super().__init__()
        self.num_plan_tokens = num_plan_tokens
        self.d_llm = d_llm
        d_flat = num_agents * d_cida

        self.projector = nn.Sequential(
            nn.Linear(d_flat, num_plan_tokens * d_llm),
            nn.LayerNorm(num_plan_tokens * d_llm),
        )

        for m in self.projector.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, state_s):
        B = state_s.size(0)
        flat = state_s.reshape(B, -1).float()
        out  = self.projector(flat)
        return out.view(B, self.num_plan_tokens, self.d_llm)


# ── Instantiate ─────────────────────────────────────────────────────────────
NUM_PLAN_TOKENS = 4

plan_projector = CIDAPlanProjector(
    d_cida          = cida_cfg.d_hidden,
    d_llm           = LLM_HIDDEN,
    num_agents      = cida_cfg.num_agents,
    num_plan_tokens = NUM_PLAN_TOKENS,
).to(DEVICE)

embed_tokens = llm.get_input_embeddings()
for p in embed_tokens.parameters():
    p.requires_grad = False

proj_params = sum(p.numel() for p in plan_projector.parameters())
print(f"✅ CIDAPlanProjector: {proj_params/1e3:.1f}K trainable params")
print(f"   num_plan_tokens={NUM_PLAN_TOKENS}, d_llm={LLM_HIDDEN}")


## 5 · Dataset — GSM8K

The dataset now returns **full answer token sequences** (`answer_ids` / `answer_mask`)
instead of a single answer token, enabling the loss to be computed over every
answer token position.

In [ ]:
import random
random.seed(42)
torch.manual_seed(42)

print("Loading GSM8K from Hugging Face…")
raw = load_dataset("openai/gsm8k", "main")
train_raw = list(raw["train"])   # 7473 examples
test_raw  = list(raw["test"])    # 1319 examples

# ─── Subset for fast training ─────────────────────────────────────────────────
random.shuffle(train_raw)
train_raw = train_raw[:TRAIN_SUBSET]

print(f"Train subset : {len(train_raw)} examples")
print(f"Test set     : {len(test_raw)} examples")
print()
print("Sample:")
print("  Q:", train_raw[0]["question"][:120])
print("  A:", train_raw[0]["answer"][:120])

In [ ]:
def extract_numeric_answer(answer_text: str) -> Optional[str]:
    """
    GSM8K answers end with '#### <number>'.
    Returns the number as a string, or None if not found.
    """
    m = re.search(r"####\s*([\-0-9,\.]+)", answer_text)
    if m:
        return m.group(1).replace(",", "").strip()
    return None


def make_prompt(question: str) -> str:
    """Chat-style prompt for Qwen1.5-Instruct."""
    return (
        "<|im_start|>system\n"
        "You are a math tutor. Solve the problem step by step, "
        "then state the final numeric answer after '####'.\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{question}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


# Quick sanity check
sample_ans = extract_numeric_answer(train_raw[0]["answer"])
print("Extracted answer:", sample_ans)

In [ ]:
# ── GSM8KDataset — Plan-and-Execute version ─────────────────────────────────
# Returns full answer token sequences so the loss covers every answer token,
# not just the first one. This gives the plan projector a much richer
# learning signal.

class GSM8KDataset(Dataset):
    """
    Tokenises the question (prompt) and the *full* gold answer separately.

    __getitem__ returns
    -------------------
    input_ids      (max_q_len,)   — prompt token IDs, left-padded
    attention_mask (max_q_len,)   — prompt padding mask
    answer_ids     (max_a_len,)   — full gold answer token IDs, right-padded
    answer_mask    (max_a_len,)   — answer padding mask (1 = real token)
    gold_num       str            — numeric answer string for exact-match eval

    Training flow
    -------------
    prompt_embeds  (B, Lq, d_llm)  ← embed_tokens(input_ids)
    plan_embeds    (B, P,  d_llm)  ← CIDAPlanProjector(cida_out)
    answer_embeds  (B, La, d_llm)  ← embed_tokens(answer_ids)
    train_embeds   (B, Lq+P+La, d_llm)  ← cat of the three

    Loss is computed only on the La answer positions.
    """

    def __init__(
        self,
        examples: list,
        tokenizer,
        max_q_len: int = 256,
        max_a_len: int = 128,
    ):
        self.tokenizer = tokenizer
        self.max_q_len = max_q_len
        self.max_a_len = max_a_len
        # Filter out examples where we cannot extract a numeric answer
        self.data = [
            ex for ex in examples
            if extract_numeric_answer(ex["answer"]) is not None
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int) -> dict:
        ex        = self.data[idx]
        prompt    = make_prompt(ex["question"])
        gold_num  = extract_numeric_answer(ex["answer"])

        # ── Prompt (question) ─────────────────────────────────────────────
        # Left-padding: causal LM needs pad on the left so the last real
        # token is always at the right end of the sequence.
        q_enc = self.tokenizer(
            prompt,
            max_length    = self.max_q_len,
            truncation    = True,
            padding       = "max_length",
            return_tensors = "pt",
        )

        # ── Full answer sequence ─────────────────────────────────────────
        # We tokenise the *complete* answer text (chain-of-thought + ####
        # number) so cross-entropy is computed over all answer tokens.
        # Right-padding is fine here because we mask out pad positions in
        # the loss computation.
        a_enc = self.tokenizer(
            ex["answer"],                # full CoT answer
            max_length    = self.max_a_len,
            truncation    = True,
            padding       = "max_length",
            return_tensors = "pt",
        )

        return {
            "input_ids"    : q_enc["input_ids"].squeeze(0),       # (Lq,)
            "attention_mask": q_enc["attention_mask"].squeeze(0), # (Lq,)
            "answer_ids"   : a_enc["input_ids"].squeeze(0),       # (La,)
            "answer_mask"  : a_enc["attention_mask"].squeeze(0),  # (La,)
            "gold_num"     : gold_num,
        }


# ── Re-build DataLoaders ─────────────────────────────────────────────────────
MAX_A_LEN = 64   # answer sequence length (full CoT answer)

train_ds = GSM8KDataset(train_raw, tokenizer, max_q_len=MAX_Q_LEN, max_a_len=MAX_A_LEN)
test_ds  = GSM8KDataset(test_raw,  tokenizer, max_q_len=MAX_Q_LEN, max_a_len=MAX_A_LEN)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_ds,  batch_size=1, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train examples : {len(train_ds)}")
print(f"Test  examples : {len(test_ds)}")
print(f"Train batches  : {len(train_loader)}")
# Shape sanity check
sample = train_ds[0]
print(f"input_ids shape  : {sample['input_ids'].shape}")
print(f"answer_ids shape : {sample['answer_ids'].shape}")
print(f"gold_num sample  : {sample['gold_num']}")


## 6 · Forward Pass — Plan-and-Execute

**Training flow** (all in embedding space; no token generation):

```
prompt_ids  ──► embed_tokens ──► prompt_embeds  (B, Lq, d_llm)
                                        │
                    last-token hidden ──► CIDA deliberation
                                        │
                               plan_projector ──► plan_embeds (B, P, d_llm)

answer_ids  ──► embed_tokens ──► answer_embeds  (B, La, d_llm)
                                        │
        cat([prompt_embeds, plan_embeds, answer_embeds], dim=1)
                                        │
                         frozen LLM (inputs_embeds=…)
                                        │
              CE loss — ONLY on answer positions (La tokens)
```

Gradients flow through: **plan_projector → CIDA** only.

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────

@torch.no_grad()
def get_prompt_hidden_and_embeds(input_ids, attention_mask):
    """Returns pooled hidden state for CIDA + raw prompt embeddings."""
    out = llm(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        use_cache=False,
    )
    last_hidden = out.hidden_states[-1]
    seq_lengths = attention_mask.sum(dim=1) - 1
    batch_idx   = torch.arange(last_hidden.size(0), device=last_hidden.device)
    pooled      = last_hidden[batch_idx, seq_lengths].float()
    prompt_embeds = embed_tokens(input_ids).detach().float()
    return pooled, prompt_embeds


def cida_forward_with_loss(input_ids, attention_mask, answer_ids, answer_mask):
    """
    Plan-and-Execute forward pass with CORRECTED loss shift.
    The logit at position (Lq+P-1) predicts the first answer token.
    """
    B, Lq = input_ids.shape
    La    = answer_ids.shape[1]
    P     = NUM_PLAN_TOKENS

    # Step 1: frozen LLM -> pooled hidden + prompt embeds
    pooled, prompt_embeds = get_prompt_hidden_and_embeds(input_ids, attention_mask)

    # Step 2: CIDA deliberation + plan projection
    cida_out    = cida(pooled=pooled)
    plan_embeds = plan_projector(cida_out["state_final"].s)

    # Step 3: embed gold answer tokens
    with torch.no_grad():
        answer_embeds = embed_tokens(answer_ids).float()

    # Step 4: concatenate [prompt | plan | answer]
    train_embeds = torch.cat([prompt_embeds, plan_embeds, answer_embeds], dim=1)

    plan_mask = torch.ones(B, P, dtype=attention_mask.dtype, device=DEVICE)
    combined_mask = torch.cat([attention_mask, plan_mask, answer_mask], dim=1)

    # Step 5: frozen LLM forward with inputs_embeds
    llm_dtype = next(llm.parameters()).dtype
    with torch.amp.autocast('cuda', dtype=llm_dtype):
        lm_out = llm(
            inputs_embeds=train_embeds.to(llm_dtype),
            attention_mask=combined_mask,
            use_cache=False,
        )
    logits = lm_out.logits.float()

    # Step 6: CE loss — CORRECTED shift
    # Logit at position (Lq+P-1) predicts first answer token answer_ids[:,0]
    # Logit at position (Lq+P+La-2) predicts last answer token answer_ids[:,La-1]
    answer_start = Lq + P
    answer_logits  = logits[:, answer_start - 1 : answer_start + La - 1, :]  # (B, La, V)
    answer_targets = answer_ids                                               # (B, La)
    loss_mask      = answer_mask.bool()                                       # (B, La)

    B_flat    = B * La
    logits_f  = answer_logits.reshape(B_flat, -1)
    targets_f = answer_targets.reshape(B_flat)
    mask_f    = loss_mask.reshape(B_flat)
    targets_f = targets_f.masked_fill(~mask_f, 0)

    loss_per_token = F.cross_entropy(logits_f, targets_f, reduction='none')
    loss = (loss_per_token * mask_f.float()).sum() / mask_f.float().sum().clamp(min=1)

    return loss, cida_out


print("✅ Plan-and-Execute forward pass defined (corrected loss shift)")
print(f"   Sequence layout: prompt({MAX_Q_LEN}) | plan({NUM_PLAN_TOKENS}) | answer({MAX_A_LEN})")


## 7 · Training Loop

In [ ]:
# ── Optimizer — CIDA + PlanProjector only ────────────────────────────────────
# embed_tokens and all LLM parameters are excluded (frozen).
trainable_params = list(cida.parameters()) + list(plan_projector.parameters())

n_trainable = sum(p.numel() for p in trainable_params)
print(f"Total trainable parameters: {n_trainable/1e6:.3f}M")
print(f"  CIDA        : {sum(p.numel() for p in cida.parameters())/1e6:.3f}M")
print(f"  PlanProjector: {sum(p.numel() for p in plan_projector.parameters())/1e3:.1f}K")

optimizer = torch.optim.AdamW(
    trainable_params,
    lr           = LR,
    weight_decay = 1e-4,
    betas        = (0.9, 0.98),   # slightly higher beta2 for stability
)

total_steps = len(train_loader) * EPOCHS
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr      = LR,
    total_steps = total_steps,
    pct_start   = 0.1,
    anneal_strategy = "cos",
)

print(f"\nTraining for {EPOCHS} epochs × {len(train_loader)} batches = {total_steps} steps")
print()

history = {"loss": [], "disagreement": [], "step": []}
global_step = 0

cida.train()
plan_projector.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    t0 = time.time()

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        answer_ids     = batch["answer_ids"].to(DEVICE)
        answer_mask    = batch["answer_mask"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        loss, cida_out = cida_forward_with_loss(
            input_ids, attention_mask, answer_ids, answer_mask
        )

        loss.backward()

        # Gradient clipping covers both CIDA and projector
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)

        optimizer.step()
        scheduler.step()

        epoch_loss  += loss.item()
        global_step += 1

        if batch_idx % 20 == 0:
            avg   = epoch_loss / (batch_idx + 1)
            disag = cida_out["disagreement"].mean().item()
            print(
                f"  Epoch {epoch+1}/{EPOCHS}  "
                f"step {batch_idx}/{len(train_loader)}  "
                f"loss={avg:.4f}  disag={disag:.3f}  "
                f"lr={scheduler.get_last_lr()[0]:.2e}"
            )
            history["loss"].append(avg)
            history["disagreement"].append(disag)
            history["step"].append(global_step)

    elapsed = time.time() - t0
    print(f"  ── Epoch {epoch+1} done in {elapsed:.0f}s, "
          f"avg loss={epoch_loss/len(train_loader):.4f}")

print("\n✅ Training complete")


## 8 · Evaluation — Autoregressive Generation

Both configurations use **full autoregressive generation** (up to 128 new tokens)
so results are directly comparable to published chain-of-thought baselines.

| Config | Generation input |
|---|---|
| **Base LLM** | `llm.generate(input_ids=prompt_ids)` — standard decoding |
| **LLM + CIDA** | `llm.generate(inputs_embeds=cat([prompt_embeds, plan_embeds]))` — plan-injected decoding |

The generated text is parsed for the `#### <number>` pattern; if absent,
the last standalone integer in the response is used as a fallback.

In [ ]:
# ── Numeric extraction & normalisation ───────────────────────────────────────

def normalise_pred(text: str) -> str:
    """Strip commas, whitespace; convert float → int where possible."""
    text = text.strip().replace(",", "")
    try:
        return str(int(float(text)))
    except ValueError:
        return text


def parse_answer_from_generation(generated_text: str) -> str:
    """
    Extracts the numeric answer from a generated string.

    Priority
    --------
    1. GSM8K convention: '#### <number>'
    2. Fallback: last integer-like token in the text
    """
    # Priority 1: standard GSM8K marker
    m = re.search(r"####\s*([\-0-9,\.]+)", generated_text)
    if m:
        return normalise_pred(m.group(1))

    # Priority 2: last standalone number (int or simple decimal)
    nums = re.findall(r"\b-?\d+(?:\.\d+)?\b", generated_text.replace(",", ""))
    if nums:
        return normalise_pred(nums[-1])

    return ""   # unparseable


# ── Base LLM evaluation (standard autoregressive generation) ─────────────────

@torch.no_grad()
def evaluate_base_llm(loader, max_new_tokens: int = 128, n_samples: int = None):
    """
    Greedy autoregressive generation from the frozen LLM (no CIDA).
    Uses standard input_ids-based generation.

    Returns exact-match accuracy on numeric answers.
    """
    llm.eval()
    correct, total = 0, 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        gold_nums      = batch["gold_num"]

        # Standard generation from token IDs
        gen_ids = llm.generate(
            input_ids       = input_ids,
            attention_mask  = attention_mask,
            max_new_tokens  = max_new_tokens,
            do_sample       = False,          # greedy
            pad_token_id    = tokenizer.pad_token_id,
            eos_token_id    = tokenizer.eos_token_id,
        )

        # Decode only the newly generated tokens (after the prompt)
        prompt_len  = input_ids.size(1)
        new_tokens  = gen_ids[:, prompt_len:]
        decoded     = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

        for pred_text, gold in zip(decoded, gold_nums):
            pred = parse_answer_from_generation(pred_text)
            if pred == normalise_pred(gold):
                correct += 1
            total += 1

        if n_samples and total >= n_samples:
            break

    return correct / total if total > 0 else 0.0


# ── CIDA + Plan-Injection evaluation ─────────────────────────────────────────

@torch.no_grad()
def evaluate_cida_llm(loader, max_new_tokens: int = 128, n_samples: int = None):
    """
    Autoregressive generation with CIDA plan tokens prepended.

    Flow
    ----
    1. Frozen LLM forward on prompt → pooled hidden state.
    2. CIDA deliberation → plan_projector → plan_embeds (B, P, d_llm).
    3. Concatenate frozen prompt_embeds and plan_embeds.
    4. llm.generate(inputs_embeds=cat_embeds) for up to 128 new tokens.
    5. Parse generated text for numeric answer.

    The plan tokens act as a latent "reasoning scaffold" inserted between
    the question and the model's generation, without altering any LLM weights.
    """
    cida.eval()
    plan_projector.eval()
    correct, total = 0, 0
    llm_dtype = next(llm.parameters()).dtype

    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        gold_nums      = batch["gold_num"]
        B              = input_ids.size(0)
        P              = NUM_PLAN_TOKENS

        # ── Step 1: contextualised hidden state for CIDA ──────────────────
        # Full LLM forward to get contextualised last-token representation
        lm_out = llm(
            input_ids            = input_ids,
            attention_mask       = attention_mask,
            output_hidden_states = True,
            use_cache            = False,
        )
        last_hidden = lm_out.hidden_states[-1]
        seq_lengths = attention_mask.sum(dim=1) - 1
        batch_idx   = torch.arange(B, device=DEVICE)
        pooled      = last_hidden[batch_idx, seq_lengths].float()  # (B, d_llm)

        # ── Step 2: CIDA deliberation → plan embeddings ───────────────────
        cida_out    = cida(pooled=pooled)
        plan_embeds = plan_projector(cida_out["state_final"].s)    # (B, P, d_llm)

        # ── Step 3: frozen prompt embeddings ─────────────────────────────
        prompt_embeds = embed_tokens(input_ids).float()             # (B, Lq, d_llm)

        # ── Step 4: concatenate prompt + plan ────────────────────────────
        # The plan tokens immediately follow the prompt in embedding space.
        # No answer embeddings: generation happens from scratch after the plan.
        cat_embeds = torch.cat(
            [prompt_embeds, plan_embeds], dim=1
        ).to(llm_dtype)   # (B, Lq + P, d_llm)

        # Build matching attention mask: all ones (no padding in plan)
        plan_mask     = torch.ones(B, P, dtype=attention_mask.dtype, device=DEVICE)
        gen_attn_mask = torch.cat([attention_mask, plan_mask], dim=1)  # (B, Lq+P)

        # ── Step 5: generate from the plan-injected embedding sequence ────
        gen_ids = llm.generate(
            inputs_embeds  = cat_embeds,
            attention_mask = gen_attn_mask,
            max_new_tokens = max_new_tokens,
            do_sample      = False,          # greedy for reproducibility
            pad_token_id   = tokenizer.pad_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

        # gen_ids shape: (B, Lq+P + new_tokens)
        # The first Lq+P tokens correspond to the inputs_embeds "prompt";
        # newly generated tokens start at index (Lq + P).
        prefix_len = cat_embeds.size(1)
        new_tokens = gen_ids[:, prefix_len:]
        decoded    = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

        for pred_text, gold in zip(decoded, gold_nums):
            pred = parse_answer_from_generation(pred_text)
            if pred == normalise_pred(gold):
                correct += 1
            total += 1

        if n_samples and total >= n_samples:
            break

    return correct / total if total > 0 else 0.0


print("✅ Evaluation functions defined")
print("   Base LLM  : llm.generate(input_ids=prompt_ids)")
print(f"  LLM+CIDA  : llm.generate(inputs_embeds=prompt+plan[{NUM_PLAN_TOKENS} tokens])")


In [ ]:
print("Running evaluation on GSM8K test set (full autoregressive generation)…")
print("(This may take 10–20 min on T4 for the full 1319 examples)\n")

t0 = time.time()
base_acc = evaluate_base_llm(test_loader, max_new_tokens=128)
print(f"  Base LLM accuracy      : {base_acc*100:.2f}%  ({time.time()-t0:.0f}s)")

t1 = time.time()
cida_acc = evaluate_cida_llm(test_loader, max_new_tokens=128)
print(f"  LLM + CIDA accuracy    : {cida_acc*100:.2f}%  ({time.time()-t1:.0f}s)")

delta = (cida_acc - base_acc) * 100
print(f"\n  Δ = {delta:+.2f} percentage points  (CIDA plan injection vs. base)")


## 9 · Results Table

In [ ]:
# Literature baselines (chain-of-thought, 0-shot or 8-shot greedy, unless noted)
# Sources: GSM8K leaderboard, Touvron et al. 2023, Zeng et al. 2024
BASELINES = [
    {"Model": "GPT-4 (OpenAI, 2023)",              "Params": "~1T",   "GSM8K %": 92.0,  "Notes": "CoT, 8-shot"},
    {"Model": "Llama-2-70B-Chat",                   "Params": "70B",   "GSM8K %": 56.8,  "Notes": "CoT, 0-shot"},
    {"Model": "Qwen1.5-7B-Chat",                    "Params": "7B",    "GSM8K %": 62.5,  "Notes": "CoT, 0-shot"},
    {"Model": "TinyLlama-1.1B-Chat",               "Params": "1.1B",  "GSM8K %": 8.3,   "Notes": "CoT, 0-shot"},
    {"Model": "Qwen1.5-1.8B-Chat (base)",          "Params": "1.8B",  "GSM8K %": 38.4,  "Notes": "CoT, 0-shot (reported)"},
    {"Model": "Qwen1.5-1.8B-Chat  ← Base (ours)", "Params": "1.8B",  "GSM8K %": round(base_acc*100, 2), "Notes": "Full CoT greedy, standard gen"},
    {"Model": "Qwen1.5-1.8B + CIDA (ours)",        "Params": "1.8B+", "GSM8K %": round(cida_acc*100, 2), "Notes": f"Full CoT greedy, CIDA plan injection"},
]

# Pretty-print the table
header = f"{'Model':<42} {'Params':>8}  {'GSM8K %':>9}  Notes"
sep    = "-" * len(header)
print(sep)
print(header)
print(sep)
for row in BASELINES:
    mark = " ◀" if "ours" in row["Model"] else ""
    print(f"{row['Model']:<42} {row['Params']:>8}  {row['GSM8K %']:>9.2f}  {row['Notes']}{mark}")
print(sep)
print()
print("NOTE: 'ours' numbers use single-token greedy prediction (first answer token).")
print("Literature numbers use full chain-of-thought generation — a stronger setup.")
print(f"CIDA improvement vs Base (same eval protocol): {delta:+.2f} pp")

## 10 · Chain-of-Thought Demo (qualitative)

Here we let the full frozen LLM **generate** a chain-of-thought answer, then do the same with CIDA's refined hidden state injected at the first decoding step.

In [ ]:
@torch.no_grad()
def generate_answer_base(question: str, max_new_tokens: int = 200) -> str:
    """Standard generation from the frozen LLM."""
    prompt = make_prompt(question)
    enc    = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out    = llm.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = out[0][enc["input_ids"].size(1):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Pick a representative test question
demo_q = test_raw[5]["question"]
demo_a = test_raw[5]["answer"]

print("═" * 70)
print("QUESTION:", demo_q)
print("═" * 70)
print("GOLD ANSWER:", demo_a[-60:])
print()

base_response = generate_answer_base(demo_q)
print("BASE LLM RESPONSE:")
print(base_response)
print()

In [ ]:
# ─── CIDA Agent Beliefs Diagnostic ────────────────────────────────────────────
# Show what each agent thinks after deliberation on the demo question

@torch.no_grad()
def show_cida_beliefs(question: str):
    prompt = make_prompt(question)
    enc    = tokenizer(prompt, return_tensors="pt",
                       max_length=MAX_Q_LEN, truncation=True).to(DEVICE)

    pooled   = get_llm_hidden(enc["input_ids"], enc["attention_mask"])
    cida_out = cida(pooled=pooled)

    b_final = cida_out["b_all"][-1][0]   # (M, K) for sample 0
    role_names = ["Prosecutor", "Defender", "Skeptic", "Integrator"]

    print(f"{'Agent':<12}  P(class=0)   P(class=1)   Uncertainty")
    print("-" * 52)
    for i, name in enumerate(role_names):
        p0 = b_final[i, 0].item()
        p1 = b_final[i, 1].item()
        print(f"  {name:<10}   {p0:.4f}       {p1:.4f}")

    p_final = cida_out["p_final"][0]
    disag   = cida_out["disagreement"][0].item()
    print("-" * 52)
    print(f"  {'CONSENSUS':<10}   {p_final[0].item():.4f}       {p_final[1].item():.4f}")
    print(f"  Disagreement across agents: {disag:.4f}")
    print(f"  Rounds used: {cida_out['rounds_used']}")


cida.eval()
print("CIDA Agent Beliefs on demo question:")
show_cida_beliefs(demo_q)

## 11 · Save CIDA Weights

In [ ]:
import os
save_dir = "cida_gsm8k_plan_weights"
os.makedirs(save_dir, exist_ok=True)

# Save CIDA config + weights
cida.save_pretrained(save_dir)

# Save the plan projector separately
torch.save(
    plan_projector.state_dict(),
    os.path.join(save_dir, "plan_projector.pt")
)

# Save results summary
results = {
    "model"          : MODEL_NAME,
    "base_acc"       : round(base_acc, 4),
    "cida_acc"       : round(cida_acc, 4),
    "delta_pp"       : round(delta, 4),
    "train_subset"   : TRAIN_SUBSET,
    "epochs"         : EPOCHS,
    "num_plan_tokens": NUM_PLAN_TOKENS,
    "cida_params"    : sum(p.numel() for p in cida.parameters()),
    "proj_params"    : sum(p.numel() for p in plan_projector.parameters()),
}
with open(os.path.join(save_dir, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"✅ Saved to '{save_dir}/'")
print(json.dumps(results, indent=2))


## 12 · Quick Reference — Reload & Architecture

```python
from cida_plugin import CIDAPlugin
cida = CIDAPlugin.from_pretrained('cida_gsm8k_plan_weights')
plan_projector.load_state_dict(
    torch.load('cida_gsm8k_plan_weights/plan_projector.pt')
)
```

---

### Plan-and-Execute Architecture

```
Question text
     │
     ▼
┌──────────────────────────┐
│  Frozen LLM (1.8B)       │  — all grads OFF
│  run ONCE on prompt      │
└────────┬─────────────────┘
         │  last-token hidden  (B, 2048)  no grad
         │  prompt_embeds      (B, Lq, 2048)  no grad
         ▼
┌──────────────────────────┐
│  CIDA Plugin (~2.7M)     │  ← TRAINED
│  4 agents × 3 rounds     │
└────────┬─────────────────┘
         │  state_final.s  (B, 4, 512)
         ▼
┌──────────────────────────┐
│  CIDAPlanProjector (~8M) │  ← TRAINED
│  Linear(2048→4×2048)     │
│  + LayerNorm             │
└────────┬─────────────────┘
         │  plan_embeds  (B, 4, 2048)  ← virtual plan tokens
         ▼
  cat([prompt_embeds, plan_embeds])       TRAINING only:
         │                                + answer_embeds
         ▼
┌──────────────────────────┐
│  Frozen LLM (again)      │  inputs_embeds= instead of input_ids=
│  generate() / forward()  │
└──────────────────────────┘
         │
   Generated answer text
   (CE loss on answer tokens during training)
```
